<a href="https://www.kaggle.com/code/aamir28/nlp-text-preprocessing-assignment?scriptVersionId=364465687" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# NLP Text Preprocessing: Pseudocode + Python on the Bollywood Movies Dataset

**Topics covered**

1. Introduction and loading the data
2. Lowercasing
3. Removing HTML tags
4. Removing URLs
5. Removing punctuation
6. Chat word treatment
7. Spelling correction
8. Removing stop words
9. Handling emojis
10. Tokenization
11. Stemming
12. Lemmatization
13. Putting it all together (full pipeline)

Each section has: **what and why**, **pseudocode**, **Python code**, and a short **observation** space for your assignment.

> Dataset: `bollywood_movies_nlp_dataset.csv` with columns `movie`, `description`, `genre`.

In [1]:
# Run once (Kaggle/Colab usually already have most of these)
!pip install -q nltk textblob emoji beautifulsoup4

In [2]:
import re, string, time
import pandas as pd
import nltk

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

pd.set_option("display.max_colwidth", 120)

## 1. Introduction

**Text preprocessing** converts raw, messy text into a clean, consistent form that a machine learning model can use.
Raw text has inconsistent casing, HTML leftovers, URLs, slang, emojis, spelling mistakes, and many words that carry little meaning.

**Pseudocode (overall idea)**
```
INPUT: raw_text
FOR each preprocessing step in chosen order:
    text = step(text)
OUTPUT: clean_text (or list of tokens)
```
The **order of steps matters**. For example, remove URLs *before* removing punctuation, otherwise the URL is broken into pieces.

In [3]:
import os
candidates = [
    "bollywood_movies_nlp_dataset.csv",
    "/kaggle/input/bollywood-movies-dataset-for-nlp/bollywood_movies_nlp_dataset.csv",
]
path = next((p for p in candidates if os.path.exists(p)), None)
if path is None:
    # Kaggle: search any attached input for the file
    for root, _, files in os.walk("/kaggle/input") if os.path.exists("/kaggle/input") else []:
        if "bollywood_movies_nlp_dataset.csv" in files:
            path = os.path.join(root, "bollywood_movies_nlp_dataset.csv")
assert path, "Upload bollywood_movies_nlp_dataset.csv next to this notebook"

df = pd.read_csv("/kaggle/input/datasets/aamir28/nlp-text-preprocessing-dataset/bollywood_movies_nlp_dataset.csv")
print(df.shape)
df.head()

(168, 3)


,movie,description,genre
0,Sholay,Two petty criminals are hired by a retired police officer to capture a ruthless dacoit terrorising a village. Friend...,"Action, Adventure"
1,Mughal-E-Azam,"Prince Salim falls in love with the court dancer Anarkali, defying his father Emperor Akbar. The epic tells a grand ...","Drama, Romance, History"
2,Mother India,A poor village woman struggles to raise her sons and uphold her principles amid hardship and debt. Her sacrifices ma...,Drama
3,Pyaasa,A disillusioned poet finds that the world values him more after his supposed death than during his life. A bitter lo...,"Drama, Musical"
4,Guide,"A tour guide helps an unhappy dancer become a star, but love and ego lead him down an unexpected spiritual path.","Drama, Romance"


In [4]:
df.info()
print(df.isna().sum())
print("Average words per description:", df["description"].str.split().str.len().mean().round(1))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   movie        168 non-null    object
 1   description  168 non-null    object
 2   genre        168 non-null    object
dtypes: object(3)
memory usage: 4.1+ KB
movie          0
description    0
genre          0
dtype: int64
Average words per description: 17.4


The dataset's descriptions are fairly clean, so for topics like HTML, URLs, chat words, emojis and spelling we also use a **noisy sample sentence** where those problems actually appear.

In [5]:
noisy = ("<p>OMG!! SRK's <b>DDLJ</b> is a MUST-WATCH movie 😍🔥 imo. "
         "Watch it at https://www.example.com/ddlj or www.imdb.com/title/tt0112870 asap!! "
         "Luv the songs, gr8 acting & the storyline is amazng, thx to the directer.</p>")
print(noisy)

<p>OMG!! SRK's <b>DDLJ</b> is a MUST-WATCH movie 😍🔥 imo. Watch it at https://www.example.com/ddlj or www.imdb.com/title/tt0112870 asap!! Luv the songs, gr8 acting & the storyline is amazng, thx to the directer.</p>


## 2. Lowercasing

**Why:** "Movie", "MOVIE" and "movie" are the same word but a computer treats them as different tokens. Lowercasing reduces vocabulary size.

**Pseudocode**
```
FUNCTION lowercase(text):
    RETURN text converted to lower case
```
**Caution:** it may lose information (e.g., "US" vs "us", "SRK" as a name).

In [6]:
def lowercase(text):
    return text.lower()

print(lowercase(noisy))
before = df["description"].str.split().explode().nunique()
after = df["description"].str.lower().str.split().explode().nunique()
print("Unique tokens before:", before, "| after lowercasing:", after)

<p>omg!! srk's <b>ddlj</b> is a must-watch movie 😍🔥 imo. watch it at https://www.example.com/ddlj or www.imdb.com/title/tt0112870 asap!! luv the songs, gr8 acting & the storyline is amazng, thx to the directer.</p>
Unique tokens before: 1222 | after lowercasing: 1210


## 3. Remove HTML tags

**Why:** Scraped web text often contains tags like `<p>`, `<br>`, `<b>` that are not part of the actual content.

**Pseudocode**
```
FUNCTION remove_html(text):
    pattern = anything starting with "<" and ending with ">"
    RETURN text with every match of pattern replaced by ""
```
Two approaches: **regex** (fast, simple) and **BeautifulSoup** (more reliable on messy HTML).

In [7]:
from bs4 import BeautifulSoup

def remove_html_regex(text):
    return re.sub(r"<.*?>", "", text)

def remove_html_bs4(text):
    return BeautifulSoup(text, "html.parser").get_text()

print("Regex:", remove_html_regex(noisy))
print("BS4  :", remove_html_bs4(noisy))

Regex: OMG!! SRK's DDLJ is a MUST-WATCH movie 😍🔥 imo. Watch it at https://www.example.com/ddlj or www.imdb.com/title/tt0112870 asap!! Luv the songs, gr8 acting & the storyline is amazng, thx to the directer.
BS4  : OMG!! SRK's DDLJ is a MUST-WATCH movie 😍🔥 imo. Watch it at https://www.example.com/ddlj or www.imdb.com/title/tt0112870 asap!! Luv the songs, gr8 acting & the storyline is amazng, thx to the directer.


## 4. Remove URLs

**Why:** Links add noise and unique tokens that don't help meaning.

**Pseudocode**
```
FUNCTION remove_urls(text):
    pattern = (http:// or https:// or www.) followed by non-space characters
    RETURN text with every match replaced by ""
```

In [8]:
def remove_urls(text):
    return re.sub(r"(https?://|www\.)\S+", "", text)

print(remove_urls(remove_html_regex(noisy)))

OMG!! SRK's DDLJ is a MUST-WATCH movie 😍🔥 imo. Watch it at  or  asap!! Luv the songs, gr8 acting & the storyline is amazng, thx to the directer.


## 5. Remove punctuation

**Why:** Characters like `! ? , . ;` rarely help bag-of-words models and create noisy tokens (`movie!` vs `movie`).

**Pseudocode**
```
FUNCTION remove_punctuation(text):
    FOR each character c in text:
        IF c is in punctuation_set: skip it
        ELSE: keep it
    RETURN kept characters
```
Three methods are compared below. Note that `str.translate` is usually the fastest.

In [9]:
punc = string.punctuation
print("Punctuation set:", punc)

def remove_punct_loop(text):
    return "".join(ch for ch in text if ch not in punc)

def remove_punct_translate(text):
    return text.translate(str.maketrans("", "", punc))

def remove_punct_regex(text):
    return re.sub(r"[^\w\s]", "", text)

sample = df["description"].iloc[0]
print(sample, "\n")
print(remove_punct_translate(sample))

# speed comparison on the whole column
for f in (remove_punct_loop, remove_punct_translate, remove_punct_regex):
    t = time.time(); df["description"].apply(f); print(f.__name__, round(time.time()-t, 4), "sec")

Punctuation set: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
Two petty criminals are hired by a retired police officer to capture a ruthless dacoit terrorising a village. Friendship, revenge and iconic dialogue define this curry-western classic. 

Two petty criminals are hired by a retired police officer to capture a ruthless dacoit terrorising a village Friendship revenge and iconic dialogue define this currywestern classic
remove_punct_loop 0.0022 sec
remove_punct_translate 0.0012 sec
remove_punct_regex 0.0009 sec


## 6. Chat word treatment

**Why:** Social media text uses abbreviations ("omg", "asap", "imo"). Expanding them gives the model the real words.

**Pseudocode**
```
chat_dict = { "omg": "oh my god", "asap": "as soon as possible", ... }

FUNCTION expand_chat_words(text):
    new_words = []
    FOR each word in text split by space:
        IF lowercase(word) is in chat_dict:
            append chat_dict[word] to new_words
        ELSE:
            append word to new_words
    RETURN new_words joined by space
```

In [10]:
chat_words = {
    "omg": "oh my god", "imo": "in my opinion", "imho": "in my humble opinion", "asap": "as soon as possible",
    "gr8": "great", "luv": "love", "thx": "thanks", "ty": "thank you", "pls": "please", "plz": "please",
    "btw": "by the way", "brb": "be right back", "idk": "i do not know", "lol": "laughing out loud",
    "fyi": "for your information", "u": "you", "ur": "your", "r": "are", "b4": "before", "2day": "today",
    "tbh": "to be honest", "smh": "shaking my head", "bff": "best friends forever", "gm": "good morning",
    "gn": "good night", "nvm": "never mind", "rofl": "rolling on the floor laughing", "w8": "wait",
    "m8": "mate", "bcoz": "because", "cuz": "because", "tmrw": "tomorrow", "msg": "message",
}

def expand_chat_words(text):
    return " ".join(chat_words.get(w.lower(), w) for w in text.split())

step = remove_punct_translate(lowercase(remove_urls(remove_html_regex(noisy))))
print("Before:", step)
print("After :", expand_chat_words(step))

Before: omg srks ddlj is a mustwatch movie 😍🔥 imo watch it at  or  asap luv the songs gr8 acting  the storyline is amazng thx to the directer
After : oh my god srks ddlj is a mustwatch movie 😍🔥 in my opinion watch it at or as soon as possible love the songs great acting the storyline is amazng thanks to the directer


## 7. Spelling correction

**Why:** Typos ("amazng", "directer") create new, rare tokens. Correcting them merges them with the right word.

**Pseudocode**
```
FUNCTION correct_spelling(text):
    corrected = []
    FOR each word in text:
        candidates = dictionary words close to word (small edit distance)
        best = candidate with highest frequency/probability
        append best to corrected
    RETURN corrected joined by space
```
We use **TextBlob**, which implements this idea (Peter Norvig style). **Warning:** it can change correct but rare words (names!), and it is slow, so apply it carefully, not blindly to the whole dataset.

In [11]:
from textblob import TextBlob

def correct_spelling(text):
    return str(TextBlob(text).correct())

text = "the storyline is amazng thanks to the directer and the acters"
print("Before:", text)
print("After :", correct_spelling(text))

# Danger: it may "fix" proper nouns
print(correct_spelling("shahrukh khan starred in sholay and devdas"))

Before: the storyline is amazng thanks to the directer and the acters
After : the storyline is amazing thanks to the directed and the actors
shahrukh khan started in holy and deeds


## 8. Removing stop words

**Why:** Words like *the, is, and, of* appear everywhere and carry little meaning for tasks like classification or search.

**Pseudocode**
```
stop_set = set of English stop words

FUNCTION remove_stopwords(text):
    RETURN words of text that are NOT in stop_set
```
**Caution:** For sentiment tasks, removing "not" or "no" changes meaning. Customise the list if needed.

In [12]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))
print(len(stop_words), "stop words, e.g.", list(stop_words)[:10])

def remove_stopwords(text, stop=stop_words):
    return " ".join(w for w in text.split() if w.lower() not in stop)

s = df["description"].iloc[1]
print("Before:", s)
print("After :", remove_stopwords(s))

# Keep negations (custom list)
custom_stop = stop_words - {"not", "no", "nor"}
print(remove_stopwords("this is not a good movie", custom_stop))

198 stop words, e.g. ["i've", 'against', 'that', 'under', 'very', 'am', 'was', 'myself', 'such', 'below']
Before: Prince Salim falls in love with the court dancer Anarkali, defying his father Emperor Akbar. The epic tells a grand story of love against royal duty.
After : Prince Salim falls love court dancer Anarkali, defying father Emperor Akbar. epic tells grand story love royal duty.
not good movie


## 9. Handling emojis

**Why:** Emojis carry emotion, but most models cannot read them as plain text. You can either **remove** them or **convert them to text** (e.g., 😍 becomes `smiling_face_with_heart-eyes`).

**Pseudocode**
```
FUNCTION remove_emoji(text):
    RETURN text with every character in emoji unicode ranges removed

FUNCTION emoji_to_text(text):
    FOR each emoji in text:
        replace it with its description (":name:")
```

In [13]:
import emoji

def remove_emoji(text):
    return emoji.replace_emoji(text, replace="")

def emoji_to_text(text):
    return emoji.demojize(text)

print("Original :", noisy[-120:])
print("Removed  :", remove_emoji(noisy)[-120:])
print("To text  :", emoji_to_text("great movie 😍🔥"))

Original : dlj or www.imdb.com/title/tt0112870 asap!! Luv the songs, gr8 acting & the storyline is amazng, thx to the directer.</p>
Removed  : dlj or www.imdb.com/title/tt0112870 asap!! Luv the songs, gr8 acting & the storyline is amazng, thx to the directer.</p>
To text  : great movie :smiling_face_with_heart-eyes::fire:


## 10. Tokenization

**Why:** Models work on units (tokens), not whole paragraphs. Tokenization splits text into **words** or **sentences**.

**Pseudocode**
```
FUNCTION word_tokenize(text):
    split text at spaces
    separate punctuation and contractions from words (don't -> do, n't)
    RETURN list of tokens

FUNCTION sentence_tokenize(text):
    split text at sentence boundaries (. ! ?) while ignoring abbreviations like "Dr."
    RETURN list of sentences
```
Compare: plain `split()`, regex, NLTK word tokenizer.

In [14]:
from nltk.tokenize import word_tokenize, sent_tokenize, regexp_tokenize

text = "Shah Rukh Khan didn't star in Sholay. Dr. Amitabh did! Isn't that amazing? It costs Rs.50 only."

print("split()       :", text.split())
print("regex \\w+     :", re.findall(r"\w+", text))
print("NLTK word     :", word_tokenize(text))
print("NLTK sentence :", sent_tokenize(text))

split()       : ['Shah', 'Rukh', 'Khan', "didn't", 'star', 'in', 'Sholay.', 'Dr.', 'Amitabh', 'did!', "Isn't", 'that', 'amazing?', 'It', 'costs', 'Rs.50', 'only.']
regex \w+     : ['Shah', 'Rukh', 'Khan', 'didn', 't', 'star', 'in', 'Sholay', 'Dr', 'Amitabh', 'did', 'Isn', 't', 'that', 'amazing', 'It', 'costs', 'Rs', '50', 'only']
NLTK word     : ['Shah', 'Rukh', 'Khan', 'did', "n't", 'star', 'in', 'Sholay', '.', 'Dr.', 'Amitabh', 'did', '!', 'Is', "n't", 'that', 'amazing', '?', 'It', 'costs', 'Rs.50', 'only', '.']
NLTK sentence : ["Shah Rukh Khan didn't star in Sholay.", 'Dr. Amitabh did!', "Isn't that amazing?", 'It costs Rs.50 only.']


In [15]:
# Tokenize the dataset
df["tokens"] = df["description"].apply(word_tokenize)
df["n_tokens"] = df["tokens"].str.len()
df[["movie", "tokens", "n_tokens"]].head(3)

,movie,tokens,n_tokens
0,Sholay,"[Two, petty, criminals, are, hired, by, a, retired, police, officer, to, capture, a, ruthless, dacoit, terrorising, ...",30
1,Mughal-E-Azam,"[Prince, Salim, falls, in, love, with, the, court, dancer, Anarkali, ,, defying, his, father, Emperor, Akbar, ., The...",29
2,Mother India,"[A, poor, village, woman, struggles, to, raise, her, sons, and, uphold, her, principles, amid, hardship, and, debt, ...",27


## 11. Stemming

**Why:** Reduces words to a root by chopping suffixes (*running, runs, ran* to *run*), shrinking the vocabulary. The result may not be a real word (*studies* to *studi*).

**Pseudocode**
```
FUNCTION stem(word):
    apply a fixed sequence of suffix-stripping rules
    (e.g., remove "ing", "ed", "ies" to "i", "ly" ...)
    RETURN stem

FUNCTION stem_text(tokens):
    RETURN [stem(t) FOR t IN tokens]
```
Common stemmers: **Porter** (classic), **Snowball** (improved), **Lancaster** (most aggressive).

In [16]:
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer

porter, snow, lanc = PorterStemmer(), SnowballStemmer("english"), LancasterStemmer()
words = ["running", "studies", "fighting", "romantic", "loved", "falling", "beautifully", "families", "wrestlers"]

pd.DataFrame({"word": words,
              "Porter": [porter.stem(w) for w in words],
              "Snowball": [snow.stem(w) for w in words],
              "Lancaster": [lanc.stem(w) for w in words]})

,word,Porter,Snowball,Lancaster
0,running,run,run,run
1,studies,studi,studi,study
2,fighting,fight,fight,fight
3,romantic,romant,romant,rom
4,loved,love,love,lov
5,falling,fall,fall,fal
6,beautifully,beauti,beauti,beauty
7,families,famili,famili,famy
8,wrestlers,wrestler,wrestler,wrestl


## 12. Lemmatization

**Why:** Like stemming, but returns a **real dictionary word** (the *lemma*) using vocabulary and grammar. *studies* gives *study*, *better* gives *good* (with POS = adjective).

**Pseudocode**
```
FUNCTION lemmatize(word, pos):
    look up word in a dictionary (WordNet) using its part of speech
    RETURN base form (lemma), or the word itself if not found

FUNCTION lemmatize_text(tokens):
    tag each token with its part of speech
    RETURN [lemmatize(t, pos(t)) FOR t IN tokens]
```
**Stemming vs lemmatization:** stemming is faster but crude; lemmatization is slower but more accurate.

In [17]:
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

lem = WordNetLemmatizer()
print(lem.lemmatize("studies"), lem.lemmatize("running"), lem.lemmatize("running", pos="v"), lem.lemmatize("better", pos="a"))

def get_wordnet_pos(tag):
    return {"J": wordnet.ADJ, "V": wordnet.VERB, "N": wordnet.NOUN, "R": wordnet.ADV}.get(tag[0], wordnet.NOUN)

def lemmatize_tokens(tokens):
    return [lem.lemmatize(w, get_wordnet_pos(t)) for w, t in pos_tag(tokens)]

toks = word_tokenize("The brothers were fighting villains while the families loved the stories")
print("Stem :", [porter.stem(t) for t in toks])
print("Lemma:", lemmatize_tokens(toks))

study running run good
Stem : ['the', 'brother', 'were', 'fight', 'villain', 'while', 'the', 'famili', 'love', 'the', 'stori']
Lemma: ['The', 'brother', 'be', 'fight', 'villain', 'while', 'the', 'family', 'love', 'the', 'story']


## 13. Full pipeline

**Pseudocode**
```
FUNCTION preprocess(text):
    text = remove_html(text)
    text = remove_urls(text)
    text = lowercase(text)
    text = emoji_to_text(text)      # or remove_emoji
    text = remove_punctuation(text) # before chat words so "omg!!" becomes "omg"
    text = expand_chat_words(text)
    tokens = word_tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize(tokens)      # or stem
    RETURN tokens
```
Spelling correction is left out of the default pipeline because it is slow and can damage names.

In [18]:
def preprocess(text, use_stemming=False):
    text = remove_html_regex(text)
    text = remove_urls(text)
    text = lowercase(text)
    text = remove_emoji(text)
    text = remove_punct_translate(text)   # before chat words so "omg!!" becomes "omg"
    text = expand_chat_words(text)
    tokens = [t for t in word_tokenize(text) if t not in stop_words]
    if use_stemming:
        return [porter.stem(t) for t in tokens]
    return lemmatize_tokens(tokens)

print(preprocess(noisy))

['oh', 'god', 'srks', 'ddlj', 'mustwatch', 'movie', 'opinion', 'watch', 'soon', 'possible', 'love', 'song', 'great', 'acting', 'storyline', 'amazng', 'thanks', 'directer']


In [19]:
df["clean_tokens"] = df["description"].apply(preprocess)
df["clean_text"] = df["clean_tokens"].str.join(" ")
df[["movie", "description", "clean_text"]].head(5)

,movie,description,clean_text
0,Sholay,Two petty criminals are hired by a retired police officer to capture a ruthless dacoit terrorising a village. Friend...,two petty criminal hire retired police officer capture ruthless dacoit terrorise village friendship revenge iconic d...
1,Mughal-E-Azam,"Prince Salim falls in love with the court dancer Anarkali, defying his father Emperor Akbar. The epic tells a grand ...",prince salim fall love court dancer anarkali defy father emperor akbar epic tell grand story love royal duty
2,Mother India,A poor village woman struggles to raise her sons and uphold her principles amid hardship and debt. Her sacrifices ma...,poor village woman struggle raise son uphold principle amid hardship debt sacrifice make symbol resilience
3,Pyaasa,A disillusioned poet finds that the world values him more after his supposed death than during his life. A bitter lo...,disillusion poet find world value suppose death life bitter look greed hypocrisy
4,Guide,"A tour guide helps an unhappy dancer become a star, but love and ego lead him down an unexpected spiritual path.",tour guide help unhappy dancer become star love ego lead unexpected spiritual path


In [20]:
from collections import Counter
all_tokens = [t for toks in df["clean_tokens"] for t in toks]
print("Vocabulary before:", df["description"].str.lower().str.split().explode().nunique())
print("Vocabulary after :", len(set(all_tokens)))
Counter(all_tokens).most_common(15)

Vocabulary before: 1210
Vocabulary after : 938


[('love', 38),
 ('man', 35),
 ('woman', 27),
 ('fall', 21),
 ('family', 19),
 ('young', 18),
 ('life', 16),
 ('two', 15),
 ('father', 13),
 ('take', 13),
 ('become', 12),
 ('find', 11),
 ('friend', 11),
 ('officer', 10),
 ('village', 10)]

## Your observations (for the assignment)

- Which step reduced the vocabulary the most?
- Did stemming or lemmatization give better-looking tokens? Why?
- Which steps could hurt a genre classifier (e.g., stop word removal, spelling correction)?
- What order of steps would you recommend, and why?

## Next steps
Convert `clean_text` to features with **Bag of Words / TF-IDF** and train a genre classifier.